In [1]:
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Optional


def compute_windowed_pearson_correlation(
    df: pd.DataFrame,
    timestamp_col: str = "time",
    sensor_cols: Optional[List[str]] = None,
    window_size: int = 100,
    step_size: int = 50,
    min_periods: int = 2
) -> List[Dict[str, Any]]:
    """
    Compute Pearson correlation matrix for each sliding window.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing timestamp + sensor variables.
    timestamp_col : str
        Name of timestamp column.
    sensor_cols : list
        Sensor variable columns. If None, auto-detect numeric columns.
    window_size : int
        Number of rows per window.
    step_size : int
        Sliding step between windows.
    min_periods : int
        Minimum observations required for correlation.

    Returns
    -------
    results : List[Dict]
        Each item contains:
        {
            'window_id': int,
            'start_index': int,
            'end_index': int,
            'start_time': ...,
            'end_time': ...,
            'correlation_matrix': pd.DataFrame
        }
    """

    df = df.copy()

    # Sort by timestamp if column exists
    if timestamp_col in df.columns:
        df = df.sort_values(timestamp_col).reset_index(drop=True)

    # Auto-detect numeric sensor columns
    if sensor_cols is None:
        sensor_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if timestamp_col in sensor_cols:
            sensor_cols.remove(timestamp_col)

    results = []
    n_rows = len(df)

    # Convert only sensor data to NumPy for efficiency
    sensor_data = df[sensor_cols]

    window_id = 0

    for start in range(0, n_rows - window_size + 1, step_size):
        end = start + window_size

        # Slice window
        window_df = sensor_data.iloc[start:end]

        # Pearson correlation matrix
        corr_matrix = window_df.corr(method="pearson", min_periods=min_periods)

        # Store structured output
        result = {
            "window_id": window_id,
            "start_index": start,
            "end_index": end - 1,
            "start_time": df.iloc[start][timestamp_col] if timestamp_col in df.columns else None,
            "end_time": df.iloc[end - 1][timestamp_col] if timestamp_col in df.columns else None,
            "correlation_matrix": corr_matrix
        }

        results.append(result)
        window_id += 1

    return results


# ==================================================
# Example Usage
# ==================================================

# Load data
df = pd.read_csv("2881821.csv")

# Run correlation windows
results = compute_windowed_pearson_correlation(
    df=df,
    timestamp_col="time",
    window_size=200,
    step_size=100
)

# Print first window correlation matrix
print("Window:", results[0]["window_id"])
print(results[0]["correlation_matrix"])


# ==================================================
# Optional: Convert all matrices into dictionary form
# ==================================================

corr_dict = {
    item["window_id"]: item["correlation_matrix"]
    for item in results
}

# Access example
print(corr_dict[0])

Window: 0
          entry_id    field1    field2    field3    field4    field5  \
entry_id  1.000000 -0.313682 -0.348353 -0.368877 -0.332879  0.999993   
field1   -0.313682  1.000000  0.036497  0.728162  0.122600 -0.313843   
field2   -0.348353  0.036497  1.000000  0.653473  0.247969 -0.348867   
field3   -0.368877  0.728162  0.653473  1.000000  0.238118 -0.369261   
field4   -0.332879  0.122600  0.247969  0.238118  1.000000 -0.333589   
field5    0.999993 -0.313843 -0.348867 -0.369261 -0.333589  1.000000   
field6    0.999993 -0.313843 -0.348867 -0.369261 -0.333589  1.000000   
field7    0.999993 -0.313843 -0.348867 -0.369261 -0.333589  1.000000   
field8    0.999993 -0.313843 -0.348867 -0.369261 -0.333589  1.000000   

            field6    field7    field8  
entry_id  0.999993  0.999993  0.999993  
field1   -0.313843 -0.313843 -0.313843  
field2   -0.348867 -0.348867 -0.348867  
field3   -0.369261 -0.369261 -0.369261  
field4   -0.333589 -0.333589 -0.333589  
field5    1.000000  1.0